## utils

In [3]:
### ~~~ GLOBAL IMPORTS ~~~ ###
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer  # type: ignore[import-untyped]
from typing import Iterable, Optional, Tuple, TypedDict
from sklearn.neighbors import BallTree  # type: ignore[import-untyped]
import pandas as pd
import numpy as np
import pathlib

### ~~~ LOCAL IMPORTS ~~~ ###
# None


### ~~~ CUSTOM TYPES ~~~ ###
class Backend(TypedDict):
    tree: BallTree
    lat_rad: np.ndarray
    lon_rad: np.ndarray
    codes: np.ndarray
    names: np.ndarray
    continents: np.ndarray
    countries: np.ndarray


### ~~~ STATE DEFINITION ~~~ ###
COLS: list[str] = [
    "Passanger_Name",  # passenger data
    "Rating",
    "Verified",
    "Review_title",
    "Review_content",
    "Traveller_Type",
    "Class",
    "Flying_Date",  # spatial data
    "Layover_Route",
    "Route",
    "Start_Location",
    "End_Location",
    "Start_Latitude",
    "Start_Longitude",
    "Start_Address",
    "End_Latitude",
    "End_Longitude",
    "End_Address",
]
COLS_PASSENGER: list[str] = COLS[0:7]
COLS_SPATIAL: list[str] = COLS[7:]
EARTH_RADIUS_KM: float = 6371.0088
K_NEIGHBORS: int = 1


def load_data(path: str) -> pd.DataFrame:
    """
    Load data from a CSV file.
    Args:
        path, str: The file path to the CSV file.
    Returns:
        df, pd.DataFrame: The loaded data as a pandas DataFrame.
    """
    ### insure the path exists ###
    if not pathlib.Path(path).exists():
        raise FileNotFoundError(f"The file at path {path} does not exist.")

    df: pd.DataFrame = pd.read_csv(path)

    return df


def get_sentiment_scores(sentence: str) -> tuple[float, int]:
    """
    Compute sentiment scores for a given sentence using VADER.
    Args:
        sentence, str: The input sentence to analyze.
    Returns:
        A tuple of the following:
            - compound sentiment score (float)
            - interpretation (int): 1 for positive, -1 for negative, 0 for neutral
    """
    ### init the analyzer ###
    analyzer: SentimentIntensityAnalyzer = SentimentIntensityAnalyzer()
    score: dict = analyzer.polarity_scores(sentence)
    interpretation: int = (
        1 if score["compound"] > 0.05 else (-1 if score["compound"] < -0.05 else 0)
    )
    return score["compound"], interpretation


def _to_radians(x: np.ndarray) -> np.ndarray:
    """Convert degrees → radians (vectorized)."""
    return np.radians(x.astype(float))


def load_airports_csv(
    path: str,
    include_types: Tuple[str, ...] = (
        "large_airport",
        "medium_airport",
    ),
) -> pd.DataFrame:
    """
    Load the OurAirports airports.csv and do minimal cleaning.
        - Reads the CSV.
        - Filters rows by 'type' ∈ include_types.
        - Drops rows with missing lat/lon.
        - Resolves a canonical 'Code' column: prefer IATA, else ICAO 'ident'.
        - Keeps only the columns we need, and adds radians for lat/lon.
    Args:
        path, str: Path to OurAirports airports.csv.
        include_types, tuple[str, ...]: Airport types to retain.
    Returns:
        df, pd.DataFrame: Cleaned airports with columns:
            ['Code','Name','Type','Latitude','Longitude','Lat_rad','Lon_rad']
    """
    ### read and copy ###
    df: pd.DataFrame = pd.read_csv(path).copy()

    ### filter on type + valid coords ###
    df = df[df["type"].isin(include_types)]
    df = df[(~df["latitude_deg"].isna()) & (~df["longitude_deg"].isna())]

    ### resolve canonical code (IATA else ICAO) ###
    code: pd.Series = np.where(
        df["iata_code"].notna() & (df["iata_code"].astype(str).str.len() > 0),
        df["iata_code"].astype(str),
        df["ident"].astype(str),
    )  # type: ignore[assignment]

    ### select + rename to your style ###
    # print(df[["continent", "iso_country"]])
    # exit()
    out: pd.DataFrame = (
        df.assign(Code=code)
        .rename(
            columns={
                "name": "Name",
                "type": "Type",
                "latitude_deg": "Latitude",
                "longitude_deg": "Longitude",
            }
        )[["Code", "Name", "Type", "Latitude", "Longitude", "continent", "iso_country"]]
        .copy()
    )

    ### radians (pure add) ###
    out["Lat_rad"] = _to_radians(out["Latitude"].to_numpy())
    out["Lon_rad"] = _to_radians(out["Longitude"].to_numpy())

    return out


def build_spatial_backend(
    airports: pd.DataFrame,
) -> Backend:
    """
    Prepare a pure, serializable description of the spatial backend.
        - If scikit-learn is available: use BallTree with haversine metric.
        - Else if SciPy is available: use cKDTree on unit sphere (XYZ).
        - Else: fallback = no tree (we’ll do brute-force haversine).
    Args:
        airports, pd.DataFrame: Output of load_airports_csv(...).
    Returns:
        backend, Backend: A functional backend descriptor with:
            {
              'tree': object-or-None,
              'lat_rad': np.ndarray,
              'lon_rad': np.ndarray,
              'codes': np.ndarray,
              'names': np.ndarray
              'continents': np.ndarray,
              'countries': np.ndarray,
            }
    """
    ### extract arrays (no mutation) ###
    lat_rad: np.ndarray = airports["Lat_rad"].to_numpy()
    lon_rad: np.ndarray = airports["Lon_rad"].to_numpy()
    codes: np.ndarray = airports["Code"].astype(str).to_numpy()
    names: np.ndarray = airports["Name"].astype(str).to_numpy()
    continents: np.ndarray = airports["continent"].astype(str).to_numpy()
    countries: np.ndarray = airports["iso_country"].astype(str).to_numpy()

    ### BallTree backend ###
    tree: BallTree = BallTree(np.c_[lat_rad, lon_rad], metric="haversine")

    ### construct the backend object ###
    backend: Backend = {
        "tree": tree,
        "lat_rad": lat_rad,
        "lon_rad": lon_rad,
        "codes": codes,
        "names": names,
        "continents": continents,
        "countries": countries,
    }

    return backend


def nearest_airport_batch(
    coords: Iterable[Tuple[float, float]],
    backend: Backend,
    max_km: Optional[float] = None,
) -> pd.DataFrame:
    """
    Vectorized batch nearest lookup.
        - Accepts an iterable of (lat, lon) in degrees.
        - Returns a DataFrame aligned to input order.
        - Applies max_km if provided (rows with no match become NaN/None).
    Args:
        coords, Iterable[(float, float)]: Sequence of query coordinates.
        backend, Backend: Output of build_spatial_backend(...).
        max_km, float|None: Optional maximum distance in km.
    Returns:
        df, pd.DataFrame: Columns [
            'Query_Lat',
            'Query_Lon',
            'Code',
            'Name',
            'Distance_km',
            'Continents',
            'Countries',
            'Index'
        ]
    Notes:
        - this methods assumes there is no NaN in coords.
    """
    ### materialize inputs (no mutation) ###
    coords_arr: np.ndarray = np.asarray(list(coords), dtype=float)
    if coords_arr.size == 0:
        return pd.DataFrame(
            columns=[
                "Query_Lat",
                "Query_Lon",
                "Code",
                "Name",
                "Continent",
                "Country",
                "Distance_km",
                "Index",
            ]
        )

    ### prepare queries in radians ###
    q_lat = coords_arr[:, 0]
    q_lon = coords_arr[:, 1]
    q_lat_rad = _to_radians(q_lat)
    q_lon_rad = _to_radians(q_lon)

    ### fetch backend data ###
    codes: np.ndarray = backend["codes"]
    names: np.ndarray = backend["names"]
    continents: np.ndarray = backend["continents"]
    countries: np.ndarray = backend["countries"]

    ### query the tree by concatenated radians ###
    dist_rad, idx = backend["tree"].query(np.c_[q_lat_rad, q_lon_rad], k=K_NEIGHBORS)
    j = idx[:, 0].astype(int)

    ### convert to km ###
    d_km = (dist_rad[:, 0] * EARTH_RADIUS_KM).astype(float)

    ### assemble result (apply radius if set) ###
    code_out = codes[j].astype(str)
    name_out = names[j].astype(str)
    continent_out = continents[j].astype(str)
    country_out = countries[j].astype(str)

    ### apply max_km if set ###
    if max_km is not None:
        mask = d_km <= max_km
        code_out = np.where(mask, code_out, None)  # type: ignore
        name_out = np.where(mask, name_out, None)  # type: ignore
        continent_out = np.where(mask, continent_out, None)  # type: ignore
        country_out = np.where(mask, country_out, None)  # type: ignore
        d_km = np.where(mask, d_km, np.nan)
        j = np.where(mask, j, -1)

    return pd.DataFrame(
        {
            "Query_Lat": q_lat,
            "Query_Lon": q_lon,
            "Code": code_out,
            "Name": name_out,
            "Continent": continent_out,
            "Country": country_out,
            "Distance_km": d_km,
            "Index": j,
        }
    )


def map_dataframe_coords_to_airport(
    df: pd.DataFrame,
    lat_col: str,
    lon_col: str,
    backend: Backend,
    prefix: str = "Nearest_",
    max_km: Optional[float] = None,
) -> pd.DataFrame:
    """
    Convenience wrapper to keep your pandas pipeline clean.
        - Does NOT mutate the input DataFrame.
        - Returns a new DataFrame with ['Nearest_Code','Nearest_Name','Nearest_Dist_km'] added.
    Args:
        df, pd.DataFrame: Input table with latitude/longitude columns.
        lat_col, str: Name of the latitude column in df (degrees).
        lon_col, str: Name of the longitude column in df (degrees).
        backend, dict: Output of build_spatial_backend(...).
        prefix, str: Prefix for the new columns.
        max_km, float|None: Optional maximum distance in km.
    Returns:
        df, pd.DataFrame: Copy of df with 3 appended columns.
    """
    ### copy the df to avoid modifying the original ###
    base = df.copy()

    ### compute batch nearest ###
    results = nearest_airport_batch(
        coords=list(zip(base[lat_col].to_list(), base[lon_col].to_list())),
        backend=backend,
        max_km=max_km,
    )

    ### merge columns in your style ###
    out = base.assign(
        **{
            f"{prefix}Code": results["Code"].to_numpy(),
            f"{prefix}Name": results["Name"].to_numpy(),
            f"{prefix}Continent": results["Continent"].to_numpy(),
            f"{prefix}Country": results["Country"].to_numpy(),
            f"{prefix}Dist_km": results["Distance_km"].to_numpy(),
        }
    )

    return out


def test() -> int:
    """"""
    path = "./dbs/raw/airports.csv"
    df_path = "./dbs/raw/db.csv"

    airports = load_airports_csv(path)
    df = pd.read_csv(df_path)
    backend = build_spatial_backend(airports)

    df_mapped = map_dataframe_coords_to_airport(
        df.dropna(subset=["Start_Latitude", "Start_Longitude"]),
        lat_col="Start_Latitude",
        lon_col="Start_Longitude",
        backend=backend,
        prefix="start_",
        max_km=100,
    )
    print(
        df_mapped[
            [
                "Start_Latitude",
                "Start_Longitude",
                "start_Code",
                "start_Name",
                "start_Dist_km",
            ]
        ].head(10)
    )
    return 0


def main() -> int:
    """"""
    ### init some stuff ###
    input_path: str = "./dbs/raw/db.csv"

    ### load the data ###
    df: pd.DataFrame = load_data(input_path)

    return 0



## Pre-processing

In [9]:
### ~~~ GLOBAL IMPORTS ~~~ ###
from matplotlib import pyplot as plt
from tqdm import tqdm
import pandas as pd


def process_passenger_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Process passenger-related data. It does the following:
        - Dense encoding of the 'Class' column. (possibly apply an exponential growth encoding later)
        - Sparse encoding of the 'Traveller_Type' column.
        - Sparse encoding of the 'Verified' column.
        - Frequency encoding of the 'Passanger_Name' column.
        - Sentiment analysis on the 'Review_content' column.
        - Drops the 'Review_title' column.
        - Drops any remaining object-type columns. (which should not be ANY)
    Args:
        df, pd.DataFrame: The input DataFrame containing passenger data.
    Returns:
        df, pd.DataFrame: The processed DataFrame.
    """
    ### copy the df to avoid modifying the original ###
    df = df.copy()

    ### do a dense enocding of the class column ###
    mapper: dict[str, int] = {
        "Economy Class": 0,
        "Business Class": 1,
        "Premium Economy": 2,
        "First Class": 3,
        "Unknown": 4,
    }
    df["Class_Encoded"] = df["Class"].map(mapper)
    df.drop(columns=["Class"], inplace=True)

    ### do a sparese encoding of the traveller type column ###
    df = pd.get_dummies(df, columns=["Traveller_Type"], prefix="Traveller")

    ### do a sparse encoding of the verified column ###
    df = pd.get_dummies(df, columns=["Verified"], prefix="Verified", drop_first=True)

    ### use the frequency encoding for the passenger name ###
    freq_encoding: pd.Series = df["Passanger_Name"].value_counts(normalize=True)
    df["Passanger_Name"] = df["Passanger_Name"].map(freq_encoding)

    ### run the sentiment analysis on the review content ###
    tqdm.pandas(desc="Computing sentiment scores")
    sentiment_scores: pd.DataFrame = (
        df["Review_content"]
        .progress_apply(
            lambda x: get_sentiment_scores(x)[0] if isinstance(x, str) else {}
        )
        .apply(pd.Series)
    )
    df = (
        pd.concat([df, sentiment_scores], axis=1)
        .drop(columns="Review_content")
        .rename(
            columns={
                0: "Sentiment_Compound",
            }
        )
    )

    ### drop the review title column ###
    """
    we do so as running the sentiment analysis on it resulted in a very sparse distribution
    """
    df.drop(columns=["Review_title"], inplace=True)

    ### make sure to drop any columns that are no longger needed ###
    object_cols = df.select_dtypes(include=["object"]).columns
    df.drop(columns=object_cols, inplace=True)

    return df


def process_spatial_data(
    df: pd.DataFrame,
    airport_path: str = "../dbs/raw/airports.csv",
) -> pd.DataFrame:
    """
    Process flight-related data, by doing the following:
        - Dropping columns
        - Mapping start and end locations to nearest airports
        - Converting layover information to a binary indicator
        - Encoding categorical variables (continent, country, airport codes)
          * using a hierarchical encoding scheme to avoid information loss
          * Do a quick sanity check to insure there is a one-one mapping with the codes
        - Dropping any remaining object-type columns.
    Args:
        df, pd.DataFrame: The input DataFrame containing flight data.
    Returns:
        df, pd.DataFrame: The processed DataFrame.
    """
    ### copy the df to avoid modifying the original ###
    df = df.copy()

    ### drop some columns ###
    """
    1) Flying_Date: The vast majority of the dates are missing, and
       cannot be extracted from other columns.
    2) Route: The route can be inferred from the start and
       end locations and any layovers, so it is redundant information.
    """
    df.drop(columns=["Flying_Date", "Route"], inplace=True)

    ### extract the airport info from the start and end locations ###
    ## 1. load the airports data ##
    df_airports: pd.DataFrame = load_airports_csv(airport_path)
    ## 2. build the spatial backend ##
    spatial_backend = build_spatial_backend(df_airports)
    ## 3. map the start location to the nearest airport ##
    df = map_dataframe_coords_to_airport(
        df=df,
        lat_col="Start_Latitude",
        lon_col="Start_Longitude",
        backend=spatial_backend,
        prefix="Start_",
    )
    ## 4. map the end location to the nearest airport ##
    df = map_dataframe_coords_to_airport(
        df=df,
        lat_col="End_Latitude",
        lon_col="End_Longitude",
        backend=spatial_backend,
        prefix="End_",
    )
    ## 5. drop the remaining spatial columns ##
    df.drop(
        columns=[
            "Start_Location",
            "End_Location",
            "Start_Address",
            "End_Address",
            "Start_Latitude",
            "Start_Longitude",
            "End_Latitude",
            "End_Longitude",
            "Start_Dist_km",
            "End_Dist_km",
            "Start_Name",  # they are redundant with the codes
            "End_Name",  # they are redundant with the codes
        ],
        inplace=True,
    )

    ### convert the Layover column to a binary indicator ###
    df["Has_Layover"] = df.Layover_Route.notna().astype(int)
    df.drop(columns=["Layover_Route"], inplace=True)

    ### Encode the Airline column ###
    ## 1. do a sparse encoding of the continent columns ##
    df = pd.get_dummies(
        df, columns=["Start_Continent"], prefix="Start_Cont", drop_first=True, dtype=int
    )
    df = pd.get_dummies(
        df, columns=["End_Continent"], prefix="End_Cont", drop_first=True, dtype=int
    )
    ## 2. do a frequency encoding of the country ##
    df["Start_Country_freq"] = df["Start_Country"].map(
        df["Start_Country"].value_counts()
    )
    df["End_Country_freq"] = df["End_Country"].map(df["End_Country"].value_counts())
    ## 3. do a frequency encoding of the airport codes ##
    df["Start_Code_freq"] = df["Start_Code"].map(df["Start_Code"].value_counts())
    df["End_Code_freq"] = df["End_Code"].map(df["End_Code"].value_counts())

    ### sanity check ###
    ## 1. get the columns of interest ##
    Start_cols_of_interest = [c for c in df.columns if c.startswith("Start_Cont_")] + [
        "Start_Country_freq",
        "Start_Code_freq",
    ]
    End_cols_of_interest = [c for c in df.columns if c.startswith("End_Cont_")] + [
        "End_Country_freq",
        "End_Code_freq",
    ]
    ## 2. check for one-to-one mapping ##
    Start_is_one_to_one = (
        df.groupby("Start_Code")[Start_cols_of_interest]
        .nunique()
        .apply(lambda s: (s <= 1).all(), axis=1)
        .all()
    )
    End_is_one_to_one = (
        df.groupby("End_Code")[End_cols_of_interest]
        .nunique()
        .apply(lambda s: (s <= 1).all(), axis=1)
        .all()
    )
    ## 3. assert the one-to-one mapping ##
    assert (
        Start_is_one_to_one
    ), "Start_Code encoding is not one-to-one with the Start code leading to information loss"
    assert (
        End_is_one_to_one
    ), "End_Code encoding is not one-to-one with the End code leading to information loss"

    ### drop any remaining object columns ###
    object_cols = df.select_dtypes(include=["object"]).columns
    df.drop(columns=object_cols, inplace=True)

    return df


def main() -> int:
    """
    This is the main function that orchestrates the data pre-processing.
        - Loads the raw data from a CSV file.
        - Drops records with missing coordinates.
        - Processes passenger-related data.
        - Processes flight-related data.
        - Concatenates the processed dataframes.
        - Saves the final processed dataframe to a CSV file.
    Args:
        None
    Returns:
        int: Exit code (0 for success).
    """
    ### init some stuff ###
    input_path: str = "../dbs/raw/db.csv"
    output_path: str = "../dbs/interm/db.csv"

    ### load the data ###
    df: pd.DataFrame = load_data(input_path)

    ### drop all records with NaN in the coords columns ###
    df.dropna(
        subset=[
            "Start_Latitude",
            "Start_Longitude",
            "End_Latitude",
            "End_Longitude",
        ],
        inplace=True,
    )

    ### process passenger data ###
    df_passenger: pd.DataFrame = process_passenger_data(df[COLS_PASSENGER])

    ### process flight data ###
    df_spatial: pd.DataFrame = process_spatial_data(df[COLS_SPATIAL])

    ### concatenate the dataframes ###
    df_final: pd.DataFrame = pd.concat([df_passenger, df_spatial], axis=1)
    df_final = df_final.astype(float)

    ### make the label binary ###
    df_final["Rating"] = (df_final["Rating"] >= 5).astype(int)

    ### drop any rows with NaN values ###
    df_final.dropna(inplace=True)

    ### write the final dataframe to a csv file ###
    df_final.to_csv(output_path, index=False)

    return 0


In [10]:
main()

Computing sentiment scores: 100%|█████████████████████████████████████████| 3416/3416 [00:13<00:00, 261.40it/s]


0

## Processing

In [14]:
### ~~~ GLOBAL IMPORTS ~~~ ###
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from typing import TypeAlias, List
import pandas as pd
import numpy as np

### ~~~ CUSTOM TYPES ~~~ ###
tensor_t: TypeAlias = np.ndarray
list_str_t: TypeAlias = List[str]

### ~~~ STATE DEFINITIONS ~~~ ###
# None


def main() -> int:
    """"""
    ### init some stuff ###
    input_path: str = "../dbs/interm/db.csv"
    output_path: str = "../dbs/cooked/data.npz"

    ### load the data ###
    df: pd.DataFrame = load_data(input_path)

    ### get the X and y ###
    X_df: pd.DataFrame = df.drop(columns=["Rating"])
    y: tensor_t = df["Rating"].to_numpy()

    ### identify column types for scaling ###
    binary_cols: list_str_t = [
        col
        for col in X_df.columns
        if col.startswith("Traveller_")
        or col.startswith("Verified_")
        or col.startswith("Start_Cont_")
        or col.startswith("End_Cont_")
        or col == "Has_Layover"
    ]
    scalable_cols: list_str_t = [col for col in X_df.columns if col not in binary_cols]

    ### define the preprocessor ###
    preprocessor: ColumnTransformer = ColumnTransformer(
        transformers=[
            ("scaler", StandardScaler(), scalable_cols),
            ("passthrough", "passthrough", binary_cols),
        ],
    )

    ### get the column names as it might be useful later ###
    y_col_name: list_str_t = ["Rating"]

    ### split the data ###
    X_train_df, X_test_df, y_train, y_test = train_test_split(
        X_df, y, test_size=0.3, random_state=42, stratify=y
    )

    ### fit and transform the data ###
    X_train: tensor_t = preprocessor.fit_transform(X_train_df)
    X_test: tensor_t = preprocessor.transform(X_test_df)

    ### get the new column order from the transformer ###
    X_col_names: list_str_t = preprocessor.get_feature_names_out()

    ### save the data as one big npz file ###
    np.savez_compressed(
        output_path,
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        X_col_names=X_col_names,
        y_col_name=y_col_name,
    )

    return 0


In [15]:
main()

0

## Run every single model

In [18]:
### ~~~ GLOBAL IMPORTS ~~~ ###
import argparse
from dataclasses import dataclass
from typing import Callable, Dict, Tuple, TypeAlias
import numpy as np
import tensorflow as tf
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

### ~~~ CUSTOM TYPES ~~~ ###
tensor_t: TypeAlias = np.ndarray


@dataclass
class EvaluationResult:
    model_name: str
    test_accuracy: float
    test_weighted_f1: float
    cross_val_accuracy: float
    cross_val_weighted_f1: float
    classification_summary: str
    confusion: tensor_t


model_runner_t: TypeAlias = Callable[
    [tensor_t, tensor_t, tensor_t, tensor_t], EvaluationResult
]

### ~~~ STATE DEFINITIONS ~~~ ###
DATA_PATH = "../dbs/cooked/data.npz"
MODEL_CHECKPOINT_PATH = "../src/models/best_model.keras"
L2_REG = 0.01
DROPOUT_RATE = 0.5
LEARNING_RATE = 0.001
EPOCHS = 1080
BATCH_SIZE = 256
VALIDATION_SPLIT = 0.2
CV_FOLDS = 5
AVAILABLE_MODELS = (
    "mlp",
    "logistic_regression",
    "random_forest",
    "gradient_boosting",
    "svm",
)

### ~~~ FUNCTION DEFINITIONS ~~~ ###


def load_data(path: str) -> Tuple[tensor_t, tensor_t, tensor_t, tensor_t]:
    """
    Load training and testing data from a NumPy binary file.

    Args:
        path: The path to the NumPy `.npz` file containing the dataset.

    Returns:
        A tuple containing the training features, training labels, testing
        features, and testing labels.
    """
    with np.load(path) as data:
        x_train = data["X_train"]
        y_train = data["y_train"]
        x_test = data["X_test"]
        y_test = data["y_test"]
    return x_train, y_train, x_test, y_test


def build_mlp_model(
    input_shape: Tuple[int, ...], l2_reg: float, dropout_rate: float
) -> Model:
    """
    Build the baseline multi-layer perceptron classifier.

    Args:
        input_shape: The shape of the input feature tensor.
        l2_reg: The L2 regularization factor to apply to dense layers.
        dropout_rate: The dropout probability for regularization.

    Returns:
        An uncompiled Keras model implementing the baseline architecture.
    """
    inputs = Input(shape=input_shape)
    x = Dense(128, activation="relu", kernel_regularizer=l2(l2_reg))(inputs)
    x = Dropout(dropout_rate)(x)
    x = Dense(64, activation="relu", kernel_regularizer=l2(l2_reg))(x)
    x = Dropout(dropout_rate)(x)
    outputs = Dense(1, activation="sigmoid")(x)
    model: Model = Model(inputs=inputs, outputs=outputs)
    return model


def compute_cross_validation_metrics(
    estimator: Pipeline, x_train: tensor_t, y_train: tensor_t
) -> Tuple[float, float]:
    """
    Compute mean cross-validation metrics for a scikit-learn estimator.

    Args:
        estimator: The estimator or pipeline to evaluate.
        x_train: The training feature tensor.
        y_train: The training label tensor.

    Returns:
        A tuple containing the mean accuracy and weighted F1-score across
        the configured cross-validation folds.
    """
    scores = cross_validate(
        estimator,
        x_train,
        y_train,
        cv=CV_FOLDS,
        scoring={"accuracy": "accuracy", "f1_weighted": "f1_weighted"},
        n_jobs=1,
    )
    cv_accuracy = float(np.mean(scores["test_accuracy"]))
    cv_weighted_f1 = float(np.mean(scores["test_f1_weighted"]))
    return cv_accuracy, cv_weighted_f1


def evaluate_predictions(
    model_name: str,
    y_true: tensor_t,
    y_pred: tensor_t,
    cross_val_accuracy: float,
    cross_val_weighted_f1: float,
) -> EvaluationResult:
    """
    Build an evaluation summary from ground truth and predicted labels.

    Args:
        model_name: A human-readable identifier for the evaluated model.
        y_true: The ground-truth target labels.
        y_pred: The predicted labels.
        cross_val_accuracy: Mean cross-validation accuracy for the model.
        cross_val_weighted_f1: Mean cross-validation weighted F1-score.

    Returns:
        A structured evaluation result containing cumulative metrics.
    """
    test_accuracy = float(accuracy_score(y_true, y_pred))
    test_weighted_f1 = float(f1_score(y_true, y_pred, average="weighted"))
    summary = classification_report(y_true, y_pred)
    matrix = confusion_matrix(y_true, y_pred)
    return EvaluationResult(
        model_name=model_name,
        test_accuracy=test_accuracy,
        test_weighted_f1=test_weighted_f1,
        cross_val_accuracy=cross_val_accuracy,
        cross_val_weighted_f1=cross_val_weighted_f1,
        classification_summary=summary,
        confusion=matrix,
    )


def train_evaluate_sklearn_pipeline(
    model_name: str,
    pipeline: Pipeline,
    x_train: tensor_t,
    y_train: tensor_t,
    x_test: tensor_t,
    y_test: tensor_t,
) -> EvaluationResult:
    """
    Train and evaluate a scikit-learn pipeline following the project protocol.

    Args:
        model_name: A descriptive identifier for the pipeline.
        pipeline: The pipeline to train and evaluate.
        x_train: Training features.
        y_train: Training labels.
        x_test: Testing features.
        y_test: Testing labels.

    Returns:
        An evaluation summary capturing cross-validation and test metrics.
    """
    cross_val_accuracy, cross_val_weighted_f1 = compute_cross_validation_metrics(
        pipeline, x_train, y_train
    )
    pipeline.fit(x_train, y_train)
    y_pred = pipeline.predict(x_test)
    return evaluate_predictions(
        model_name,
        y_test,
        y_pred,
        cross_val_accuracy,
        cross_val_weighted_f1,
    )


def build_logistic_regression_pipeline() -> Pipeline:
    """
    Construct the logistic regression pipeline with feature scaling.

    Returns:
        A scikit-learn pipeline combining standard scaling and logistic regression.
    """
    classifier = LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    )
    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("classifier", classifier),
        ]
    )
    return pipeline


def build_random_forest_pipeline() -> Pipeline:
    """
    Construct the random forest classification pipeline.

    Returns:
        A scikit-learn pipeline wrapping the configured random forest classifier.
    """
    classifier = RandomForestClassifier(
        n_estimators=400,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )
    pipeline = Pipeline(steps=[("classifier", classifier)])
    return pipeline


def build_gradient_boosting_pipeline() -> Pipeline:
    """
    Construct the gradient boosting classification pipeline.

    Returns:
        A scikit-learn pipeline wrapping the gradient boosting classifier.
    """
    classifier = GradientBoostingClassifier(
        learning_rate=0.05,
        n_estimators=300,
        max_depth=3,
        subsample=0.8,
        random_state=42,
    )
    pipeline = Pipeline(steps=[("classifier", classifier)])
    return pipeline


def build_svm_pipeline() -> Pipeline:
    """
    Construct the support vector machine classification pipeline.

    Returns:
        A scikit-learn pipeline combining feature scaling with an SVM classifier.
    """
    classifier = SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        probability=True,
        random_state=42,
    )
    pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("classifier", classifier),
        ]
    )
    return pipeline


def run_mlp_model(
    x_train: tensor_t,
    y_train: tensor_t,
    x_test: tensor_t,
    y_test: tensor_t,
) -> EvaluationResult:
    """
    Train and evaluate the baseline multi-layer perceptron classifier.

    Args:
        x_train: Training features.
        y_train: Training labels.
        x_test: Testing features.
        y_test: Testing labels.

    Returns:
        An evaluation result describing the MLP performance.
    """
    model = build_mlp_model(x_train.shape[1:], L2_REG, DROPOUT_RATE)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )
    model_checkpoint = ModelCheckpoint(
        MODEL_CHECKPOINT_PATH,
        save_best_only=True,
        monitor="val_loss",
    )
    model.fit(
        x_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        callbacks=[early_stopping, model_checkpoint],
        verbose=0,
    )
    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)
    predictions = model.predict(x_test, verbose=0)
    y_pred = (predictions.flatten() >= 0.5).astype(int)
    result = evaluate_predictions("MLP", y_test, y_pred, float(np.nan), float(np.nan))
    result.test_accuracy = float(accuracy)
    result.test_weighted_f1 = float(f1_score(y_test, y_pred, average="weighted"))
    return result


def run_logistic_regression_model(
    x_train: tensor_t,
    y_train: tensor_t,
    x_test: tensor_t,
    y_test: tensor_t,
) -> EvaluationResult:
    """
    Train and evaluate the logistic regression model.

    Args:
        x_train: Training features.
        y_train: Training labels.
        x_test: Testing features.
        y_test: Testing labels.

    Returns:
        The evaluation summary for logistic regression.
    """
    pipeline = build_logistic_regression_pipeline()
    return train_evaluate_sklearn_pipeline(
        "Logistic Regression", pipeline, x_train, y_train, x_test, y_test
    )


def run_random_forest_model(
    x_train: tensor_t,
    y_train: tensor_t,
    x_test: tensor_t,
    y_test: tensor_t,
) -> EvaluationResult:
    """
    Train and evaluate the random forest model.

    Args:
        x_train: Training features.
        y_train: Training labels.
        x_test: Testing features.
        y_test: Testing labels.

    Returns:
        The evaluation summary for random forest classification.
    """
    pipeline = build_random_forest_pipeline()
    return train_evaluate_sklearn_pipeline(
        "Random Forest", pipeline, x_train, y_train, x_test, y_test
    )


def run_gradient_boosting_model(
    x_train: tensor_t,
    y_train: tensor_t,
    x_test: tensor_t,
    y_test: tensor_t,
) -> EvaluationResult:
    """
    Train and evaluate the gradient boosting model.

    Args:
        x_train: Training features.
        y_train: Training labels.
        x_test: Testing features.
        y_test: Testing labels.

    Returns:
        The evaluation summary for gradient boosting classification.
    """
    pipeline = build_gradient_boosting_pipeline()
    return train_evaluate_sklearn_pipeline(
        "Gradient Boosting", pipeline, x_train, y_train, x_test, y_test
    )


def run_svm_model(
    x_train: tensor_t,
    y_train: tensor_t,
    x_test: tensor_t,
    y_test: tensor_t,
) -> EvaluationResult:
    """
    Train and evaluate the support vector machine model.

    Args:
        x_train: Training features.
        y_train: Training labels.
        x_test: Testing features.
        y_test: Testing labels.

    Returns:
        The evaluation summary for SVM classification.
    """
    pipeline = build_svm_pipeline()
    return train_evaluate_sklearn_pipeline(
        "Support Vector Machine", pipeline, x_train, y_train, x_test, y_test
    )


def get_model_registry() -> Dict[str, model_runner_t]:
    """
    Build the registry mapping model identifiers to execution functions.

    Returns:
        A dictionary linking model keys to their respective runner functions.
    """
    return {
        "mlp": run_mlp_model,
        "logistic_regression": run_logistic_regression_model,
        "random_forest": run_random_forest_model,
        "gradient_boosting": run_gradient_boosting_model,
        "svm": run_svm_model,
    }


def display_evaluation_result(result: EvaluationResult) -> None:
    """
    Pretty-print the evaluation metrics for a trained model.

    Args:
        result: The evaluation summary to display.
    """
    cross_val_accuracy = (
        "N/A"
        if np.isnan(result.cross_val_accuracy)
        else f"{result.cross_val_accuracy:.4f}"
    )
    cross_val_weighted_f1 = (
        "N/A"
        if np.isnan(result.cross_val_weighted_f1)
        else f"{result.cross_val_weighted_f1:.4f}"
    )
    print("=" * 80)
    print(f"Model: {result.model_name}")
    print(f"Test Accuracy: {result.test_accuracy:.4f}")
    print(f"Test Weighted F1: {result.test_weighted_f1:.4f}")
    print(f"CV Accuracy: {cross_val_accuracy}")
    print(f"CV Weighted F1: {cross_val_weighted_f1}")
    print("Classification Report:")
    print(result.classification_summary)
    print("Confusion Matrix:")
    print(result.confusion)


def main() -> None:
    """
    Main script execution function selecting, training, and evaluating models.
    """
    x_train, y_train, x_test, y_test = load_data(DATA_PATH)
    model_registry = get_model_registry()
    for model_key in get_model_registry().keys():
        runner = model_registry[model_key]
        result = runner(x_train, y_train, x_test, y_test)
        display_evaluation_result(result)


In [19]:
main()

Model: MLP
Test Accuracy: 0.8068
Test Weighted F1: 0.8066
CV Accuracy: N/A
CV Weighted F1: N/A
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.76      0.80       533
           1       0.77      0.86      0.81       492

    accuracy                           0.81      1025
   macro avg       0.81      0.81      0.81      1025
weighted avg       0.81      0.81      0.81      1025

Confusion Matrix:
[[405 128]
 [ 70 422]]
Model: Logistic Regression
Test Accuracy: 0.8000
Test Weighted F1: 0.7996
CV Accuracy: 0.8075
CV Weighted F1: 0.8071
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.74      0.79       533
           1       0.75      0.86      0.81       492

    accuracy                           0.80      1025
   macro avg       0.80      0.80      0.80      1025
weighted avg       0.81      0.80      0.80      1025

Confusion Matrix:
[[395 138]
 [ 67 425]]
Model: